In [11]:
# ==============================================================
# 03 - MODELO CON PREPROCESADO DE TEXTO (TF-IDF) + SVM (LinearSVC)
# Usa E_PRGM_ACADEMICO como TEXTO y F_ESTRATOVIVIENDA como CATEGÓRICA
# ==============================================================

!pip -q install unidecode

# ------------------------------
# 0) dependencias
# ------------------------------
import os, joblib
import numpy as np
import pandas as pd
from unidecode import unidecode

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 4.7 MB/s eta 0:00:00


In [12]:
# ------------------------------
# 1) constantes
# ------------------------------
TARGET   = "RENDIMIENTO_GLOBAL"
COL_TXT  = "E_PRGM_ACADEMICO"   # texto
COL_CAT  = "F_ESTRATOVIVIENDA"  # categórica
FEATURES = [COL_TXT, COL_CAT]

In [13]:
# ------------------------------
# 2) utilidades de limpieza (MISMA normalización que en otros .ipynb)
# ------------------------------
def norm(s):
    """Normaliza texto: quita tildes, trim, mayúsculas y colapsa espacios."""
    if pd.isna(s):
        return s
    t = unidecode(str(s)).strip().upper()
    return " ".join(t.split())

# TF-IDF recibe 1-D de strings. Este paso convierte (n,1) -> (n,)
def squeeze_1d(X):
    X = np.asarray(X).ravel()
    return X

In [14]:
# ------------------------------
# 3) cargar y limpiar train
# ------------------------------
assert os.path.exists("train.csv"), "No se encontró train.csv"
df = pd.read_csv("train.csv")

# nos quedamos SOLO con las columnas de interés
df = df[FEATURES + [TARGET]].copy()

# normalización exacta de ambas columnas de entrada
df[COL_TXT] = df[COL_TXT].apply(norm)
df[COL_CAT] = df[COL_CAT].apply(norm)

# split estratificado
X = df[FEATURES]
y = df[TARGET]

x_tr, x_va, y_tr, y_va = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

In [15]:
# ------------------------------
# 4) pipelines con IMPUTACIÓN para evitar NaNs
# ------------------------------
txt_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="")),
    ("squeeze", FunctionTransformer(squeeze_1d, validate=False)),
    ("tfidf",   TfidfVectorizer(min_df=5, ngram_range=(1,2)))
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", sparse_output=True))
])

# IMPORTANTÍSIMO: pasar SIEMPRE listas de columnas en el ColumnTransformer
pre = ColumnTransformer(transformers=[
    ("txt", txt_pipe, [COL_TXT]),
    ("cat", cat_pipe, [COL_CAT]),
])

# modelo completo: preprocesamiento + clasificador
pipe = Pipeline(steps=[
    ("pre", pre),
    ("clf", LinearSVC())
])

In [16]:
# ------------------------------
# 5) entrenamiento y métricas
# ------------------------------
pipe.fit(x_tr, y_tr)
y_hat = pipe.predict(x_va)

acc = accuracy_score(y_va, y_hat)
f1m = f1_score(y_va, y_hat, average="macro")
print(f"Accuracy validación: {acc:.4f}")
print(f"F1-macro validación: {f1m:.4f}")
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_va, y_hat))

Accuracy validación: 0.3813
F1-macro validación: 0.3732

=== Reporte de Clasificación ===
              precision    recall  f1-score   support

        alto       0.48      0.57      0.52       510
        bajo       0.41      0.40      0.40       498
  medio-alto       0.31      0.20      0.24       497
  medio-bajo       0.29      0.36      0.32       512

    accuracy                           0.38      2017
   macro avg       0.38      0.38      0.37      2017
weighted avg       0.38      0.38      0.37      2017



In [17]:
# ------------------------------
# 6) guardar artefacto
# ------------------------------
os.makedirs("models", exist_ok=True)
out_model = "models/03_pipeline_svm_tfidf.pkl"
joblib.dump(pipe, out_model)
print(f"\nGuardado: {out_model}")


Guardado: models/03_pipeline_svm_tfidf.pkl


In [18]:
# ------------------------------
# 7) función de SUBMISIÓN (formato Kaggle)
#    Aplica la MISMA normalización y usa el pipeline guardado
# ------------------------------
def make_submission_03(
    path_test="test.csv",
    path_sample="submission_example.csv",
    path_model="models/03_pipeline_svm_tfidf.pkl",
    out_path="submission_03_svm.csv",
    id_col="ID"
):
    assert os.path.exists(path_test),   f"No existe {path_test}"
    assert os.path.exists(path_sample), f"No existe {path_sample}"
    assert os.path.exists(path_model),  f"No existe {path_model}"

    test   = pd.read_csv(path_test)
    sample = pd.read_csv(path_sample)

    # Normalización EXACTA de columnas de entrada
    # (si alguna no existe, fallará claramente)
    test[COL_TXT] = test[COL_TXT].apply(norm)
    test[COL_CAT] = test[COL_CAT].apply(norm)

    # cargar pipeline entrenado y predecir
    pipe = joblib.load(path_model)
    yhat = pipe.predict(test[[COL_TXT, COL_CAT]])

    # respetar encabezado del sample
    cols_out   = list(sample.columns)
    target_col = cols_out[1]

    sub = pd.DataFrame({
        cols_out[0]: test[id_col] if id_col in test.columns else sample[cols_out[0]],
        target_col:  yhat
    })
    sub.to_csv(out_path, index=False)
    print(f"Archivo de submission creado: {out_path}")
    return sub

# (opcional) generar una vez
_ = make_submission_03()

Archivo de submission creado: submission_03_svm.csv
